# 第 5 章 · 次方与 e

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/05-power-and-e.html) · 仓库里 `transformer-deep-dive/notes/lessons/05-power-and-e.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 次方 | $2^3, 2^2, \dots, 2^{-2}$ | `2.0 ** torch.arange(3, -3, -1)` | [8, 4, 2, 1, 0.5, 0.25] |
| 利滚利 | $(1+\tfrac1n)^n$ | `(1 + 1 / n) ** n` | n = 365：2.71457；n = 100 万：2.71828 |
| e | e ≈ 2.71828 | `math.e`、`torch.exp(torch.tensor(1.0))` | 2.718281828…、2.7182817 |
| $e^{x}$ | $e^{2}, e^{1}, e^{0}, e^{-1}, e^{-2}$ | `torch.exp(x)` | [7.389, 2.718, 1, 0.368, 0.135] |
| 增长速度 = 自己 | $(e^x)' = e^x$ | `torch.exp(t).backward()` | x = 1 处斜率 2.718 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import torch

**函数 `powers_of_two`**：2^3, 2^2, 2^1, 2^0, 2^-1, 2^-2：每往右一格除以 2。

In [ ]:
def powers_of_two() -> torch.Tensor:
    """2^3, 2^2, 2^1, 2^0, 2^-1, 2^-2：每往右一格除以 2。"""
    return 2.0 ** torch.arange(3, -3, -1)

**函数 `compound`**：存 1 元、年利率 100%、一年结息 n 次：(1 + 1/n)^n。

In [ ]:
def compound(n: int) -> float:
    """存 1 元、年利率 100%、一年结息 n 次：(1 + 1/n)^n。"""
    return (1 + 1 / n) ** n

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("2 ** torch.arange(3, -3, -1) =", powers_of_two().tolist())
print("利滚利:")
for n in [1, 2, 4, 12, 365, 1_000_000]:
    print(f"  一年 {n:>7} 次 → {compound(n):.5f}")
print("math.e =", math.e, "  torch.exp(1) =", float(torch.exp(torch.tensor(1.0))))
x = torch.tensor([2.0, 1.0, 0.0, -1.0, -2.0])
print("x       =", x.tolist())
print("e^x     =", [round(float(v), 3) for v in torch.exp(x)])
print("e^x 都 > 0:", bool((torch.exp(torch.linspace(-50, 50, 101)) > 0).all()))
print("e^x 增长速度 = 自己（导数）：在 x = 1 处斜率 ≈", end=" ")
t = torch.tensor(1.0, requires_grad=True)
torch.exp(t).backward()
print(round(float(t.grad), 3), "= e^1")

## 3. 代码里才看得到的

1. **float32 只有大约 7 位有效数字**：`math.e` 是 2.718281828459045，`torch.exp(1)` 是 2.7182817。模型里用的就是这种精度，够用。
2. **`requires_grad` 和 `backward()`**（requires gradient：需要 / 梯度）：标上 `requires_grad=True`，PyTorch 会记住这个数参与了哪些计算；`backward()` 一键算出"它变一点点，结果变多少"（导数），放在 `.grad` 里。这里验证了 $e^x$ 的增长速度等于它自己。训练就靠这个，续篇第 2 章细讲。

## 4. 自己改一改

- 把 `t` 改成 2.0，`.grad` 应该是多少？先猜再跑。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 几次方就是几个自己相乘；0 次方等于 1，负次方是小数，永远不为负。
> e 是利滚利的上限，约 2.718；$e^{x}$ 永远是正数，x 大一点涨很多；代码写 exp(x)。